<div id="singlestore-header" style="display: flex; background-color: rgba(255, 167, 103, 0.25); padding: 5px;">
    <div id="icon-image" style="width: 90px; height: 90px;">
        <img width="100%" height="100%" src="https://raw.githubusercontent.com/singlestore-labs/spaces-notebooks/master/common/images/header-icons/pipeline.png" />
    </div>
    <div id="text" style="padding: 5px; margin-left: 10px;">
        <div id="badge" style="display: inline-block; background-color: rgba(0, 0, 0, 0.15); border-radius: 4px; padding: 4px 8px; align-items: center; margin-top: 6px; margin-bottom: -2px; font-size: 80%">SingleStore Notebooks</div>
        <h1 style="font-weight: 500; margin: 8px 0 0 4px;">Extend Real-Time Ingestion with Kafka Connect</h1>
    </div>
</div>

<div class="alert alert-block alert-warning">
    <b class="fa fa-solid fa-exclamation-circle"></b>
    <div>
        <p><b>Note</b></p>
        <p>You need a Standard or Premium workspace, a database user with the <tt>SUPER</tt> permission (to switch on connector pipelines, an experimental feature), and an AWS account that can create an Amazon Kinesis data stream. The stream costs a few cents an hour and is deleted in the clean-up step.</p>
    </div>
</div>

The goal of this notebook is to demonstrate how **Kafka Connect pipelines** let SingleStore read directly from streaming sources such as **Amazon Kinesis**, with no Kafka cluster, Connect workers or other intermediate infrastructure in between.

**The story:** AdVenture Inc. (yes, the ad startup from our AI and search notebook) just bought **PixelPop Games**, maker of the hit mobile game *Pigeon Panic*. Every ad shown, tapped or turned into an in-game purchase flows into Amazon Kinesis. Morgan, the CMO, wants those events next to the ad campaigns by Friday. The platform team wants six weeks to stand up Kafka.

<table>
  <tr><th>Chapter</th><th>What happens</th><th>You use</th></tr>
  <tr><td>1</td><td>The acquisition: game events start flowing into Kinesis</td><td>Amazon Kinesis, the Digital Marketing dataset</td></tr>
  <tr><td>2</td><td>No middlemen: Kinesis straight into SingleStore</td><td><span style="background:#d199ff; color:#1b1b1b; font-weight:700; border-radius:4px; padding:2px 8px; font-size:85%;">NEW</span> Kafka Connect pipeline (<code>LOAD DATA KAFKACONNECT</code>)</td></tr>
  <tr><td>3</td><td>The live scoreboard: which ads make money</td><td>Background pipeline + SQL on fresh data</td></tr>
  <tr><td>4</td><td>Oops, replay it</td><td><span style="background:#d199ff; color:#1b1b1b; font-weight:700; border-radius:4px; padding:2px 8px; font-size:85%;">NEW</span> <code>PIPELINES_SOURCE_OFFSETS</code>, <span style="background:#d199ff; color:#1b1b1b; font-weight:700; border-radius:4px; padding:2px 8px; font-size:85%;">NEW</span> <code>ALTER PIPELINE ... SET OFFSETS</code></td></tr>
  <tr><td>5</td><td>Same trick, other clouds</td><td>Google Cloud Pub/Sub connector pipeline</td></tr>
</table>

## Prerequisites

**1. Generate the Digital Marketing dataset.** The dataset can be generated using the website here: [digital-marketing.labs.singlestore.com](https://digital-marketing.labs.singlestore.com/dashboard)

1. Make sure your workspace is running.
2. Open the website and fill in the connection form: your workspace host with `https://` in front (find it under **Connect** in the portal), username `admin`, your workspace password, and database name `martech`. Click **Connect**.
3. Click **Setup Database** and keep the **Simulator** switch on.
4. Under **Locations**, tick a few cities. Each city adds ad campaigns.
5. Wait until **Ad Campaigns** shows a number above 0. Ad notifications start flowing a little later.

If you skip this, the notebook makes up advertisers and ad events from the dataset's vendor list instead.

**2. Get AWS access keys.** Create an IAM user (or use an existing one) with this policy, and create an access key for it:

```json
{
  "Version": "2012-10-17",
  "Statement": [{
    "Effect": "Allow",
    "Action": ["kinesis:CreateStream", "kinesis:DeleteStream", "kinesis:DescribeStream",
               "kinesis:DescribeStreamSummary", "kinesis:ListShards", "kinesis:PutRecords",
               "kinesis:GetRecords", "kinesis:GetShardIterator"],
    "Resource": "arn:aws:kinesis:*:*:stream/pixelpop-*"
  }]
}
```

The policy only covers streams whose names start with `pixelpop-`. Use a dedicated user: the pipeline stores the keys in its configuration.

## Chapter 0: Setup

Start with the database drop-down at the top of the notebook empty. The setup cell creates a database called `adventure_streams`.

In [1]:
%pip install singlestoredb pandas boto3 --quiet

In [2]:
import base64, getpass, json, os, random, re, time, uuid
import pandas as pd
import singlestoredb as s2

pd.set_option('display.max_colwidth', 120)
random.seed(7)

DB_NAME = 'adventure_streams'
DEMO_DB = 'martech'          # created by the Digital Marketing demo website
PIPELINE = 'pixelpop_events'  # also the name of the table the pipeline creates

USE_DB = None

def connect():
    global conn
    conn = s2.connect(database=USE_DB) if USE_DB else s2.connect()
    conn.autocommit(True)
    if USE_DB:
        with conn.cursor() as cur:
            cur.execute(f'USE {USE_DB}')

def _query(query, params=None):
    with conn.cursor() as cur:
        cur.execute(query, params)
        if cur.description:
            cols = [d[0] for d in cur.description]
            return pd.DataFrame(cur.fetchall(), columns=cols)
    return None

def run_sql(query, params=None):
    # Run a query and return a DataFrame (or None). Reconnects once if an idle connection was dropped.
    try:
        return _query(query, params)
    except Exception as e:
        if getattr(e, 'errno', None) in (2006, 2013) or 'Lost connection' in str(e) or 'gone away' in str(e):
            print('(Connection was dropped, reconnecting...)')
            connect()
            return _query(query, params)
        raise

def try_sql(query, params=None, label='This step'):
    # Like run_sql(), but explains and continues if the workspace doesn't support something.
    try:
        return run_sql(query, params)
    except Exception as e:
        print(f'{label} was skipped: {e}')
        return None

connect()
run_sql(f'CREATE DATABASE IF NOT EXISTS {DB_NAME}')
USE_DB = DB_NAME
connect()
print('Connected to database:', run_sql('SELECT DATABASE() AS db').iloc[0, 0])
print('SingleStore version:  ', run_sql('SELECT @@memsql_version AS v').iloc[0, 0])

<div class="alert alert-block alert-warning">
    <b class="fa fa-solid fa-exclamation-circle"></b>
    <div>
        <p><b>Action Required</b></p>
        <p>Select <tt>adventure_streams</tt> from the database drop-down at the top of this notebook (refresh the list if it isn't there yet).</p>
    </div>
</div>

<div style="border: 2px solid #d199ff; border-radius: 6px; padding: 8px 14px; margin: 6px 0;">
<span style="background:#d199ff; color:#1b1b1b; font-weight:700; border-radius:4px; padding:2px 8px; font-size:85%;">NEW</span> &nbsp;<b>Kafka Connect pipelines (experimental)</b><br/>Connector pipelines run Kafka Connect source connectors inside SingleStore. They're switched on with an experimental-features setting. The next cell adds it to whatever is already set, instead of overwriting it. This needs the <tt>SUPER</tt> permission and applies to the whole workspace.
</div>

In [3]:
FLAG = 'connector_pipelines_enabled=true'
current = run_sql("SHOW VARIABLES LIKE 'experimental_features_config'")
value = current.iloc[0, 1] if current is not None and len(current) else ''
if FLAG in (value or ''):
    print('Connector pipelines are already enabled.')
else:
    new_value = f'{value},{FLAG}' if value else FLAG
    run_sql('SET GLOBAL experimental_features_config = %s', (new_value,))
    print('Connector pipelines enabled:', run_sql("SHOW VARIABLES LIKE 'experimental_features_config'").iloc[0, 1])

## Chapter 1: The acquisition

<div style="border-left: 4px solid #d199ff; background: rgba(209,153,255,0.10); padding: 8px 14px;">
<b>#war-room</b> <span style="opacity:0.7">Mon 9:00 AM</span> &nbsp;<b>Morgan (CMO):</b> We bought PixelPop Games! 5 million players see our ads inside <i>Pigeon Panic</i>. Their events live in something called Kinesis. I want to know which ads make money. By Friday.
</div>

<div style="border-left: 4px solid #d199ff; background: rgba(209,153,255,0.10); padding: 8px 14px;">
<b>#war-room</b> <span style="opacity:0.7">Mon 9:04 AM</span> &nbsp;<b>Priya (Platform Lead):</b> Kinesis to our database? We'd need a Kafka cluster, Kafka Connect workers and a schema registry. Six weeks, minimum.
</div>

First, the ad campaigns from the Digital Marketing demo, plus recent ad notifications to replay as in-game ad impressions:

In [4]:
has_demo = run_sql('''
    SELECT COUNT(*) AS n FROM information_schema.TABLES
    WHERE TABLE_SCHEMA = %s AND TABLE_NAME IN ('offers', 'notifications', 'cities')
''', (DEMO_DB,)).iloc[0, 0] == 3

if (has_demo
        and run_sql(f'SELECT COUNT(*) FROM {DEMO_DB}.offers').iloc[0, 0] > 0
        and run_sql(f'SELECT COUNT(*) FROM {DEMO_DB}.notifications').iloc[0, 0] > 0):
    offers = run_sql(f'SELECT offer_id, customer AS advertiser, notification_content AS ad_text FROM {DEMO_DB}.offers')
    shown = run_sql(f'''
        SELECT n.subscriber_id AS player_id, n.offer_id, n.cost_cents, c.city_name AS city
        FROM {DEMO_DB}.notifications n JOIN {DEMO_DB}.cities c ON c.city_id = n.city_id
        ORDER BY n.ts DESC LIMIT 20000
    ''')
    print(f'Using the demo database: {len(offers)} ad campaigns, {len(shown)} recent ad notifications.')
else:
    print('No demo data found, so making up campaigns from the dataset vendor list.')
    vendors = pd.read_json('https://raw.githubusercontent.com/singlestore-labs/demo-realtime-digital-marketing/main/data/vendors.json')
    vendors = vendors.drop_duplicates('vendor').head(300)
    offers = pd.DataFrame({
        'offer_id': range(1, len(vendors) + 1),
        'advertiser': vendors['vendor'].values,
        'ad_text': [f'{random.randint(10, 50)}% off at {v}' for v in vendors['vendor']],
    })
    shown = pd.DataFrame({
        'player_id': [random.randint(1, 100000) for _ in range(20000)],
        'offer_id': [random.choice(offers['offer_id'].tolist()) for _ in range(20000)],
        'cost_cents': [random.randint(2, 15) for _ in range(20000)],
        'city': 'New York',
    })

# Keep a copy of the campaigns in our own database for the scoreboard joins
run_sql('DROP TABLE IF EXISTS campaigns')
run_sql('CREATE TABLE campaigns (offer_id BIGINT PRIMARY KEY, advertiser VARCHAR(100), ad_text TEXT)')
with conn.cursor() as cur:
    cur.executemany('INSERT INTO campaigns VALUES (%s, %s, %s)', offers.values.tolist())
offers.head()

Now connect to AWS. Your keys are typed in, not saved in the notebook:

In [5]:
import boto3

AWS_REGION = os.environ.get('AWS_REGION') or input('AWS region [us-east-1]: ').strip() or 'us-east-1'
KINESIS_REGION = AWS_REGION.upper().replace('-', '_')
AWS_KEY_ID = os.environ.get('AWS_ACCESS_KEY_ID') or getpass.getpass('AWS access key ID: ')
AWS_SECRET = os.environ.get('AWS_SECRET_ACCESS_KEY') or getpass.getpass('AWS secret access key: ')

kinesis = boto3.client('kinesis', region_name=AWS_REGION,
                       aws_access_key_id=AWS_KEY_ID, aws_secret_access_key=AWS_SECRET)

STREAM = f'pixelpop-events-{uuid.uuid4().hex[:6]}'
SHARDS = 2
kinesis.create_stream(StreamName=STREAM, ShardCount=SHARDS)
print(f'Creating Kinesis stream {STREAM} with {SHARDS} shards...')
kinesis.get_waiter('stream_exists').wait(StreamName=STREAM)
print('Stream is active.')

Each ad notification becomes an in-game **impression**. About 1 in 8 players tap the ad (**click**), and about a third of those buy something (**purchase**). The events are JSON, just as a mobile game would send them:

In [6]:
campaign_ids = offers['offer_id'].tolist()

def game_events(n_impressions, from_real=True):
    events = []
    sample = shown.sample(n_impressions, replace=len(shown) < n_impressions) if from_real else None
    for i in range(n_impressions):
        row = sample.iloc[i] if from_real else None
        base = {
            'game': 'Pigeon Panic',
            'player_id': int(row['player_id']) if from_real else random.randint(1, 100000),
            'offer_id': int(row['offer_id']) if from_real else random.choice(campaign_ids),
            'city': str(row['city']) if from_real else 'New York',
            'level': random.randint(1, 60),
        }
        cost = int(row['cost_cents']) if from_real else random.randint(2, 15)
        now = time.time() * 1000
        events.append({**base, 'event_id': uuid.uuid4().hex, 'event': 'impression', 'cost_cents': cost, 'ts_ms': now})
        if random.random() < 0.12:
            events.append({**base, 'event_id': uuid.uuid4().hex, 'event': 'click', 'ts_ms': now + 800})
            if random.random() < 0.33:
                events.append({**base, 'event_id': uuid.uuid4().hex, 'event': 'purchase',
                               'amount_cents': random.choice([99, 199, 499, 999, 1999]), 'ts_ms': now + 5000})
    return events

def send(events):
    for i in range(0, len(events), 500):     # PutRecords takes up to 500 records per call
        batch = events[i:i + 500]
        kinesis.put_records(StreamName=STREAM, Records=[
            {'Data': json.dumps(e).encode(), 'PartitionKey': str(e['player_id'])} for e in batch])
    return len(events)

print(f'Sent {send(game_events(3000))} events to Kinesis.')

## Chapter 2: No middlemen

<div style="border-left: 4px solid #d199ff; background: rgba(209,153,255,0.10); padding: 8px 14px;">
<b>#war-room</b> <span style="opacity:0.7">Mon 10:15 AM</span> &nbsp;<b>Morgan (CMO):</b> Priya says six weeks. You have until lunch.
</div>

<div style="border: 2px solid #d199ff; border-radius: 6px; padding: 8px 14px; margin: 6px 0;">
<span style="background:#d199ff; color:#1b1b1b; font-weight:700; border-radius:4px; padding:2px 8px; font-size:85%;">NEW</span> &nbsp;<b>Kafka Connect pipeline for Amazon Kinesis</b><br/>A Kafka Connect source connector runs <i>inside</i> the pipeline, so SingleStore reads Kinesis directly. No Kafka brokers, no Connect workers. <tt>CREATE INFERRED PIPELINE</tt> also creates the destination table for you.
</div>

In [7]:
config = {
    'connector.class': 'com.github.jcustenborder.kafka.connect.kinesis.KinesisSourceConnector',
    'kafka.topic': 'pixelpop',          # a label only: no Kafka topic is needed
    'kinesis.stream': STREAM,
    'kinesis.region': KINESIS_REGION,
    'kinesis.position': 'TRIM_HORIZON', # start from the oldest record in the stream
    'tasks.max': SHARDS,                # one task per shard
}
credentials = {
    'aws.access.key.id': AWS_KEY_ID,
    'aws.secret.key.id': AWS_SECRET,
}

try_sql(f'DROP PIPELINE IF EXISTS {PIPELINE}', label='Removing an old pipeline')
run_sql(f'DROP TABLE IF EXISTS {PIPELINE}')
run_sql(f'''
    CREATE INFERRED PIPELINE {PIPELINE}
    AS LOAD DATA KAFKACONNECT 'kafka-connector'
    CONFIG %s
    CREDENTIALS %s
    FORMAT AVRO
''', (json.dumps(config), json.dumps(credentials)))
print(f'Pipeline {PIPELINE} created. That was the whole integration.')

<div class="alert alert-block alert-info"><b class="fa fa-solid fa-flask"></b><div>
<p><b>Evaluator's corner: what you didn't have to build.</b> Compare this with the usual route from Kinesis to a database:</p>
</div></div>

<table>
  <tr><th>Usual setup</th><th>With a Kafka Connect pipeline</th></tr>
  <tr><td>Kafka cluster (brokers, storage, upgrades)</td><td>Not needed</td></tr>
  <tr><td>Kafka Connect worker cluster</td><td>Not needed: the connector runs in the pipeline</td></tr>
  <tr><td>A sink connector and its configuration</td><td>Not needed: SingleStore is the destination</td></tr>
  <tr><td>Monitoring for each hop</td><td>One place: <code>SHOW PIPELINES</code> and <code>information_schema</code></td></tr>
</table>

Run the pipeline once in the foreground. The connector needs a few seconds to start, so the cell checks a few times:

In [8]:
for attempt in range(6):
    try_sql(f'START PIPELINE {PIPELINE} FOREGROUND', label='Foreground run')
    loaded = run_sql(f'SELECT COUNT(*) FROM {PIPELINE}').iloc[0, 0]
    if loaded:
        break
    time.sleep(5)
print(f'{loaded} events loaded from Kinesis.')
run_sql(f'SHOW CREATE TABLE {PIPELINE}').iloc[0, 1]

Each row holds one Kinesis record in the `record` column. Peek at one:

In [9]:
sample = json.loads(run_sql(f'SELECT record FROM {PIPELINE} LIMIT 1').iloc[0, 0])
print(json.dumps(sample, indent=2)[:1200])

The game's JSON sits inside the Kinesis record's data field. The next cell finds it, works out how it's encoded, and creates a `game_events` view with proper columns, so every later query reads like a normal table:

In [10]:
data_key = next((k for k in sample if k.lower() == 'data'), None)
if data_key is None:
    raise KeyError(f'No data field found in the Kinesis record. Fields: {list(sample)}')
value = sample[data_key]

if isinstance(value, dict):
    payload = f"JSON_EXTRACT_JSON(record, '{data_key}')"
else:
    try:
        json.loads(value)
        payload = f"JSON_EXTRACT_STRING(record, '{data_key}')"
    except (ValueError, TypeError):
        payload = f"FROM_BASE64(JSON_EXTRACT_STRING(record, '{data_key}'))"
print('Game event JSON is read with:', payload)

run_sql('DROP VIEW IF EXISTS game_events')
run_sql(f'''
    CREATE VIEW game_events AS
    SELECT JSON_EXTRACT_STRING({payload}, 'event_id')                    AS event_id,
           JSON_EXTRACT_STRING({payload}, 'event')                       AS event,
           FROM_UNIXTIME(JSON_EXTRACT_DOUBLE({payload}, 'ts_ms') / 1000) AS ts,
           JSON_EXTRACT_BIGINT({payload}, 'player_id')                   AS player_id,
           JSON_EXTRACT_BIGINT({payload}, 'offer_id')                    AS offer_id,
           JSON_EXTRACT_STRING({payload}, 'city')                        AS city,
           JSON_EXTRACT_BIGINT({payload}, 'level')                       AS level,
           JSON_EXTRACT_BIGINT({payload}, 'cost_cents')                  AS cost_cents,
           JSON_EXTRACT_BIGINT({payload}, 'amount_cents')                AS amount_cents
    FROM {PIPELINE}
''')
run_sql('SELECT event, COUNT(*) AS events FROM game_events GROUP BY event ORDER BY events DESC')

## Chapter 3: The live scoreboard

<div style="border-left: 4px solid #d199ff; background: rgba(209,153,255,0.10); padding: 8px 14px;">
<b>#war-room</b> <span style="opacity:0.7">Mon 11:58 AM</span> &nbsp;<b>Morgan (CMO):</b> Lunch is in two minutes. Which ads actually make money? And I want to watch it update LIVE.
</div>

Start the pipeline in the background so it keeps pulling from Kinesis, then keep the game busy for a minute. Watch the totals climb as events land:

In [11]:
run_sql(f'START PIPELINE {PIPELINE}')

for tick in range(6):
    send(game_events(500))
    time.sleep(10)
    counts = run_sql('''
        SELECT IFNULL(SUM(event = 'impression'), 0) AS impressions, IFNULL(SUM(event = 'click'), 0) AS clicks,
               IFNULL(SUM(event = 'purchase'), 0) AS purchases
        FROM game_events
    ''').iloc[0]
    print(f'+{(tick + 1) * 10}s  impressions {int(counts.impressions):>6}  '
          f'clicks {int(counts.clicks):>5}  purchases {int(counts.purchases):>4}')

Now the question Morgan asked: **return on ad spend (ROAS)** per advertiser, computed from Kinesis events joined with the ad campaigns, in one query on live data:

In [12]:
run_sql('''
    SELECT c.advertiser,
           ANY_VALUE(c.ad_text)                                   AS ad,
           SUM(e.event = 'impression')                            AS impressions,
           SUM(e.event = 'click')                                 AS clicks,
           ROUND(100 * SUM(e.event = 'click') / NULLIF(SUM(e.event = 'impression'), 0), 1) AS ctr_pct,
           SUM(IFNULL(e.cost_cents, 0)) / 100                     AS spend_usd,
           SUM(IFNULL(e.amount_cents, 0)) / 100                   AS revenue_usd,
           ROUND(SUM(IFNULL(e.amount_cents, 0)) / NULLIF(SUM(IFNULL(e.cost_cents, 0)), 0), 1) AS roas
    FROM game_events e JOIN campaigns c ON c.offer_id = e.offer_id
    GROUP BY c.advertiser
    HAVING impressions >= 5
    ORDER BY roas DESC
    LIMIT 10
''')

And where players are when they buy, by city and game level:

In [13]:
run_sql('''
    SELECT city,
           CASE WHEN level <= 20 THEN 'levels 1-20' WHEN level <= 40 THEN 'levels 21-40' ELSE 'levels 41-60' END AS stage,
           COUNT(*) AS purchases,
           SUM(amount_cents) / 100 AS revenue_usd
    FROM game_events
    WHERE event = 'purchase'
    GROUP BY city, stage
    ORDER BY revenue_usd DESC
    LIMIT 10
''')

<div style="border-left: 4px solid #d199ff; background: rgba(209,153,255,0.10); padding: 8px 14px;">
<b>#war-room</b> <span style="opacity:0.7">Mon 12:01 PM</span> &nbsp;<b>Morgan (CMO):</b> Six weeks?! It's 12:01. I'm buying everyone lunch. Except Priya.
</div>

## Chapter 4: Oops, replay it

<div style="border-left: 4px solid #d199ff; background: rgba(209,153,255,0.10); padding: 8px 14px;">
<b>#war-room</b> <span style="opacity:0.7">Tue 4:30 PM</span> &nbsp;<b>Morgan (CMO):</b> Someone on my team ran <code>TRUNCATE</code> on the events. Tell me the data isn't gone.
</div>

<div style="border: 2px solid #d199ff; border-radius: 6px; padding: 8px 14px; margin: 6px 0;">
<span style="background:#d199ff; color:#1b1b1b; font-weight:700; border-radius:4px; padding:2px 8px; font-size:85%;">NEW</span> &nbsp;<b>PIPELINES_SOURCE_OFFSETS</b><br/>Shows how far a Kafka Connect pipeline has read in its source. For Kinesis, that's a position per shard.
</div>

In [14]:
offsets = try_sql(f'''
    SELECT * FROM information_schema.PIPELINES_SOURCE_OFFSETS
    WHERE DATABASE_NAME = DATABASE() AND PIPELINE_NAME = '{PIPELINE}'
''', label='PIPELINES_SOURCE_OFFSETS')
offsets

Kinesis keeps records for 24 hours by default, so the events are still in the stream. Simulate the accident, rewind the pipeline to the start of the stream, and load everything again:

<div style="border: 2px solid #d199ff; border-radius: 6px; padding: 8px 14px; margin: 6px 0;">
<span style="background:#d199ff; color:#1b1b1b; font-weight:700; border-radius:4px; padding:2px 8px; font-size:85%;">NEW</span> &nbsp;<b>ALTER PIPELINE ... SET OFFSETS for Kafka Connect pipelines</b><br/>Moves a Kafka Connect pipeline's read position, for example back to the earliest record, to replay data.
</div>

In [15]:
before = run_sql(f'SELECT COUNT(*) FROM {PIPELINE}').iloc[0, 0]
run_sql(f'STOP PIPELINE {PIPELINE}')
run_sql(f'TRUNCATE TABLE {PIPELINE}')
print(f'Oops: {before} events deleted. Rows left: {run_sql(f"SELECT COUNT(*) FROM {PIPELINE}").iloc[0, 0]}')

try_sql(f'ALTER PIPELINE {PIPELINE} SET OFFSETS EARLIEST', label='Rewinding the pipeline')
for attempt in range(6):
    try_sql(f'START PIPELINE {PIPELINE} FOREGROUND', label='Replay run')
    after = run_sql(f'SELECT COUNT(*) FROM {PIPELINE}').iloc[0, 0]
    if after >= before:
        break
    time.sleep(5)
print(f'Replayed from Kinesis: {after} of {before} events back.')

<div style="border-left: 4px solid #d199ff; background: rgba(209,153,255,0.10); padding: 8px 14px;">
<b>#war-room</b> <span style="opacity:0.7">Tue 4:36 PM</span> &nbsp;<b>Morgan (CMO):</b> I take back everything I said about databases.
</div>

## Chapter 5: Same trick, other clouds

Kinesis is one connector. The same framework reads Google Cloud Pub/Sub with a connector that comes prepackaged, using the same pattern. If PixelPop's next acquisition runs on Google Cloud, the pipeline looks like this (shown for reference, not run here):

```sql
CREATE INFERRED PIPELINE pubsub_pipeline
AS LOAD DATA KAFKACONNECT 'CloudPubSubSourceConnector'
CONFIG '{
  "kafka.topic": "pubsub-topic",
  "cps.project": "my-gcp-project",
  "cps.subscription": "singlestore-events-sub",
  "tasks.max": "1"
}'
CREDENTIALS '{"gcp.credentials.json": "<service-account-key-json>"}'
FORMAT JSON;
```

Same `CREATE INFERRED PIPELINE`, same `START PIPELINE`, same offsets view. Only the connector and its settings change.

## What you built

- A <span style="background:#d199ff; color:#1b1b1b; font-weight:700; border-radius:4px; padding:2px 8px; font-size:85%;">NEW</span> **Kafka Connect pipeline** that reads Amazon Kinesis directly, with no Kafka or Connect cluster in between
- A table created for you by `CREATE INFERRED PIPELINE`, and a view that turns raw records into columns
- A live return-on-ad-spend scoreboard joining streaming game events with ad campaigns
- Replay after an accident with <span style="background:#d199ff; color:#1b1b1b; font-weight:700; border-radius:4px; padding:2px 8px; font-size:85%;">NEW</span> `PIPELINES_SOURCE_OFFSETS` and <span style="background:#d199ff; color:#1b1b1b; font-weight:700; border-radius:4px; padding:2px 8px; font-size:85%;">NEW</span> `ALTER PIPELINE ... SET OFFSETS`

## Clean up

Stops and drops the pipeline (which removes the stored AWS keys from SingleStore), drops the `adventure_streams` database and deletes the Kinesis stream. The demo's `martech` database is left alone.

In [16]:
try_sql(f'STOP PIPELINE {PIPELINE}', label='Stop pipeline')
try_sql(f'DROP PIPELINE IF EXISTS {PIPELINE}', label='Drop pipeline')
try_sql(f'DROP DATABASE IF EXISTS {DB_NAME}', label='Drop database')
try:
    kinesis.delete_stream(StreamName=STREAM, EnforceConsumerDeletion=True)
    print(f'Deleting Kinesis stream {STREAM}.')
except Exception as e:
    print('Could not delete the stream, delete it in the AWS console:', e)
print('All clean. Priya will never know.')

<div id="singlestore-footer" style="background-color: rgba(194, 193, 199, 0.25); height:2px; margin-bottom:10px"></div>
<div><img src="https://raw.githubusercontent.com/singlestore-labs/spaces-notebooks/master/common/images/singlestore-logo-grey.png" style="padding: 0px; margin: 0px; height: 24px"/></div>